# Questao 4 - Dados 
Problemática: alguns produtos podem ter sido vendidos abaixo do custo, possivelmente por erro operacional.
Causa:

    O sistema de vendas (vendas_2023_2024.csv) registra valores em real BRL
    O catálogo de fornecedores (custos_importacao.json) registra custos unitários em dollar USD
    O câmbio varia diariamente 

Objetivo:Cruzar o custo em dólar do dia da venda com o valor de venda em reais
para identificar onde houve prejuízo real.



### 1. Importação de Bibliotecas e Requisição

In [1]:
# importação de bibliotecas
import requests
import pandas as pd
import sqlite3

In [2]:
# URL da API do Banco Central com dados de 20 até 2025.
url_bcb = "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='01-04-2016'&@dataFinalCotacao='12-31-2025'&$top=10000&$format=json&$select=cotacaoCompra,cotacaoVenda,dataHoraCotacao"

In [3]:
# Fazendo a requisição
response = requests.get(url_bcb)
dados_json = response.json()

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [4]:
#visuallização inicial
dados_json

NameError: name 'dados_json' is not defined

### 2. Extração dos valores

In [ ]:
# Extraindo os dados a partir da chave 'value'
df_cambio = pd.DataFrame(dados_json['value'])


In [ ]:
# Visualizando as primeiras linhas para conferência
print("Cotações carregadas com sucesso:")
df_cambio.head()

### 3. Padronização da data para cruzamento de dados

In [ ]:
# Convertendo para formato de data e extraindo apenas o dia
df_cambio['data_ajustada'] = pd.to_datetime(df_cambio['dataHoraCotacao']).dt.date

df_cambio['data_ajustada'].info()

In [ ]:
# Renomeando a coluna cotacaoVenda para facilitar o cálculo do prejuízo
df_cambio = df_cambio.rename(columns={'cotacaoVenda': 'taxa_venda_bcb'})

# Mantendo apenas as colunas necessárias para a LH Nautical
df_cambio = df_cambio[['data_ajustada', 'taxa_venda_bcb']]


In [ ]:
df_cambio.sample(5)

### 4. Codigo de Integração das APIs e Cálculo de prejuízo

In [ ]:
# Carregar e padronizar as datas de vendas (Resolvendo a inconsistência crítica)
df_vendas = pd.read_csv('datasets/vendas_2023_2024.csv')

# Tentamos converter as datas; onde falhar (por formatos mistos), usamos dayfirst=True
df_vendas['sale_date_dt'] = pd.to_datetime(df_vendas['sale_date'], format='mixed', dayfirst=False)
df_vendas['sale_date_dt'] = df_vendas['sale_date_dt'].dt.date
df_vendas.to_csv("dataset/vendas_2023_2024.csv")

df_vendas

In [ ]:
# 2. Carregar o arquivo de custos normalizado (da Questão 3)
df_custos = pd.read_csv('datasets/custos_importacao_normalizado.csv')
df_custos['start_date'] = pd.to_datetime(df_custos['start_date']).dt.date
df_custos

###  Questões 4.1, 4.2 & 4.3


In [ ]:
# Criar conexão SQL em memória
conn = sqlite3.connect(':memory:')

# Salvar tudo como tabelas SQL
df_vendas.to_sql('vendas', conn, index=False, if_exists='replace')
df_custos.to_sql('custos', conn, index=False, if_exists='replace')
df_cambio.to_sql('cambio', conn, index=False, if_exists='replace')

print("Tabelas SQL (vendas, custos e cambio) prontas para consulta.")

In [ ]:
query_financeiro = """
WITH Vendas_Com_Cambio AS (
    -- Une vendas com a cotação do BCB do dia exato
    SELECT 
        v.id_product,
        v.qtd,
        v.total AS receita_brl,
        v.sale_date,
        c.taxa_venda_bcb
    FROM vendas v
    JOIN cambio c ON v.sale_date = c.data_ajustada
),
Custo_Vigente AS (
    -- Calcula o custo BRL usando o custo USD unitário vigente na data
    SELECT 
        vcc.*,
        (SELECT usd_price 
         FROM custos 
         WHERE product_id = vcc.id_product 
           AND start_date <= vcc.sale_date
         ORDER BY start_date DESC LIMIT 1) AS usd_price_unitario
    FROM Vendas_Com_Cambio vcc
),
Calculo_Prejuizo AS (
    -- custo_total_brl = (usd_price * taxa_venda * qtd)
    SELECT 
        id_product,
        receita_brl,
        (usd_price_unitario * taxa_venda_bcb * qtd) AS custo_total_brl
    FROM Custo_Vigente
)
SELECT 
    id_product,
    SUM(receita_brl) AS receita_total,
    SUM(CASE WHEN receita_brl < custo_total_brl THEN (custo_total_brl - receita_brl) ELSE 0 END) AS prejuizo_total,
    (SUM(CASE WHEN receita_brl < custo_total_brl THEN (custo_total_brl - receita_brl) ELSE 0 END) / SUM(receita_brl)) * 100 AS percentual_perda
FROM Calculo_Prejuizo
GROUP BY id_product
ORDER BY prejuizo_total DESC;
"""

In [ ]:
# Executando a consulta e exibindo o resultado
df_resultado_sql = pd.read_sql(query_financeiro, conn)
conn.close()
df_resultado_sql

In [ ]:
df_resultado_sql.iloc[df_resultado_sql['percentual_perda'].idxmax()]

#### ***Importante***

Utilizei a data de 04 de Abril de 2016 até 31 de Dezembro de 2025, que são as datas mínimas e máximas de vendas.O prejuízo foi contabilizado como receita - custo, isso éTotalVendidoEmReais - (precoUnitario * quantidadeComprada * taxa de cambio)Quanto as suposições: Como o catálogo de fornecedores possui preços que variam no tempo, assumi que o custo unitário válido para a transação é aquele com a start_date mais recente em relação à data da venda.Para vendas ocorridas em finais de semana ou feriados (onde não há cotação oficial), a lógica de cruzamento buscou a última cotação disponível imediatamente anterior.
Escopo Financeiro: Conforme solicitado, ignorei impostos e fretes, focando exclusivamente na margem bruta entre o custo de aquisição em dólar e o preço de venda final.